# Laya release triage: fine-tuned Laya vs. tabular and text baselines

This notebook runs the experiment fixed in
[PREREGISTRATION.md](https://github.com/kambasana/56/blob/research/laya-proper/experiments/release-triage/PREREGISTRATION.md).
Step by step instructions: [RUN-IN-COLAB.md](https://github.com/kambasana/56/blob/research/laya-proper/experiments/release-triage/RUN-IN-COLAB.md).

**Before you run it** (Colab Pro+):

1. *Runtime → Change runtime type*: **A100 GPU**, and **High-RAM** if that option is offered. If the dialog shows a
   *Background execution* toggle, turn it on.
2. Optional secrets (key icon in the left bar, each with **Notebook access** on): `GH_TOKEN` (live progress on
   GitHub), `HF_TOKEN` (Hugging Face read token, only against download rate limits).
3. Options are in the **Settings** form below (no code edits needed).
4. *Runtime → Run all* (or press ▶ on Settings, then on **Run**), and allow Google Drive access.

**What you see:** one control panel under the Run cell: an overall progress bar with an ETA, the ten steps (Setup →
GPU → Storage → Install → Data → Checks → Models → Training runs → Verdict → Finish), the current run's progress, a
table of the ten runs, a plain-language line saying what is happening now, Drive / GitHub / heartbeat indicators,
any errors (with where the full traceback is), a collapsible log, and at the end the verdict and the summary table.

Once the panel shows the first run training, you may close the tab (with Pro+ background execution). Training runs
in a detached process on the VM, independent of the browser. Everything is written to the VM's fast local disk
(`/content/work`) and copied to `MyDrive/laya-release-triage/<run tag>/` (write to a temporary file, then rename)
after every stage and run, and small files every ~2 minutes. At the end the notebook syncs and verifies everything
on Drive (sha256, including `results.zip`), pushes a last progress update to GitHub, flushes and unmounts Drive and
then, if `AUTO_RELEASE_RUNTIME` is on, **releases the runtime so no more compute units are spent**. If anything failed
to sync or verify, the runtime is kept and the panel says why.

**Resume after a disconnect:** *Runtime → Reconnect* (same GPU type if offered), then *Runtime → Run all* (or ▶ on
Run). Finished runs are verified (sha256) and skipped; a training process that is still alive is re-attached and the
panel is redrawn from the ledger. If the panel stopped moving after the browser reconnected, press ■ on the Run cell
and then ▶ again: training keeps running in the background. A run that was mid-training when the VM was lost
restarts from its base checkpoint (laya 0.4.1 cannot resume a run mid-way).

The code of every stage is in [`colab/lrt_runner.py`](https://github.com/kambasana/56/blob/research/laya-proper/experiments/release-triage/colab/lrt_runner.py),
downloaded at a pinned commit and checked by sha256 before it runs. This is execution and presentation plumbing
only: no experiment rule, data, threshold or pass condition depends on it.

**Remote control by the assistant** (owner-approved; switch off with `ALLOW_REMOTE_CONTROL` in Settings): with
`GH_TOKEN` set, the notebook checks `control.json` in its run folder on branch `results/laya-colab` about once a
minute and accepts only pause / resume (between runs), stop_now, retry / skip / rescore one run, ping and a
diagnostic dump. It cannot run code or change any rule, data or setting; every command is acknowledged in
`control-ack.jsonl` and shown in the panel. Details: RUN-IN-COLAB.md, "Remote control by the assistant".

**Send back:** `results.zip` (or the `results` folder) from `MyDrive/laya-release-triage/<run tag>/`.

Shai-Hulud is one incident family among many here. Every recall figure is a macro average over families, and
no gate names a family.

In [ ]:
#@title Settings { display-mode: "form" }
#@markdown Set the options here, then *Runtime → Run all*. Nothing else needs editing.
#@markdown
#@markdown **RUN_TAG**: leave empty to continue the run recorded in `MyDrive/laya-release-triage/run_tag.txt` (a new one is started the first time). A tag continues or starts that run.
RUN_TAG = ""  #@param {type:"string"}
#@markdown **RETRY_FAILED**: give runs recorded as failed another 3 attempts.
RETRY_FAILED = False  #@param {type:"boolean"}
#@markdown **AUTO_RELEASE_RUNTIME**: at the very end, once everything is synced to Drive and verified, release the GPU runtime so no more compute units are spent.
AUTO_RELEASE_RUNTIME = True  #@param {type:"boolean"}
#@markdown **PUSH_PROGRESS**: push live progress to GitHub (only if the `GH_TOKEN` secret exists and has notebook access).
PUSH_PROGRESS = True  #@param {type:"boolean"}
#@markdown **SMOKE**: quick end-to-end check on a tiny slice (1 epoch) under its own `smoke-...` run tag. Not the experiment.
SMOKE = False  #@param {type:"boolean"}
#@markdown **ALLOW_REMOTE_CONTROL**: let the assistant send a few fixed commands through the results branch on GitHub (pause/resume between runs, stop now, retry/skip/rescore one run, ping, diagnostic dump). No code, no shell, no change to any rule, data or setting; every command is acknowledged on GitHub. Needs `GH_TOKEN`. Untick to switch it off.
ALLOW_REMOTE_CONTROL = True  #@param {type:"boolean"}
#@markdown ---
#@markdown Normally left as they are:
EXPORT_ONNX = True  #@param {type:"boolean"}
RUN_POSITIVE_CONTROL = True  #@param {type:"boolean"}
RUN_FINETUNE = True  #@param {type:"boolean"}
RUN_ZERO_SHOT = True  #@param {type:"boolean"}
USE_DRIVE = True  #@param {type:"boolean"}
STALL_MINUTES = 30  #@param {type:"integer"}
OUTPUT_DIR = ""  #@param {type:"string"}

In [ ]:
#@title ▶ Run { display-mode: "form" }
#@markdown Runs every step in order under one control panel (progress, current run, runs table, errors, log and, at
#@markdown the end, the verdict). Running this cell again resumes: finished work is verified and kept, and a training
#@markdown process that is still running is re-attached. If the panel stops moving after a reconnect: ■ then ▶.
# Pinned inputs (PREREGISTRATION.md): the dataset commit and the sha256 of its MANIFEST.json, and the runner module
# colab/lrt_runner.py (every stage's code) at a fixed commit, checked against its sha256 before it is imported.
DATA_REF = "afcb46d3754fd865269de34707abfb75e83f3dd5"
MANIFEST_SHA256 = "164de05ca30b9503fbf88f8ae8427d2e0e1ba73cc516312be9b5b2c8661d2674"
RUNNER_REF = "f5f80d34bd6bd77008f38e23f24d721f910f4b75"
RUNNER_SHA256 = "99f9388306fe4b94e54f6fec8e61c396dc8ce3b5bb0736070ae206c0e53a7717"
import hashlib, os, pathlib, sys, tempfile, time, types, urllib.request


def _lrt_runner():
    """colab/lrt_runner.py at RUNNER_REF: a verified local copy if there is one, else downloaded; sha256-checked."""
    local = pathlib.Path(tempfile.gettempdir()) / f"lrt_runner-{RUNNER_SHA256[:16]}.py"
    src = os.environ.get("LRT_RUNNER_FILE")   # tests only: a local copy, held to the same sha256
    data = pathlib.Path(src or local).read_bytes() if (src or local.exists()) else b""
    if hashlib.sha256(data).hexdigest() != RUNNER_SHA256 and not src:
        url = f"https://raw.githubusercontent.com/kambasana/56/{RUNNER_REF}/experiments/release-triage/colab/lrt_runner.py"
        for i in range(4):
            try:
                with urllib.request.urlopen(url, timeout=60) as r:
                    data = r.read()
                break
            except Exception as e:
                if i == 3:
                    raise SystemExit(f"could not download the runner ({type(e).__name__}); check the connection and "
                                     "run this cell again")
                time.sleep(3 * (i + 1))
    if hashlib.sha256(data).hexdigest() != RUNNER_SHA256:
        raise SystemExit("colab/lrt_runner.py does not match its pinned sha256: nothing was run")
    local.write_bytes(data)
    mod = types.ModuleType("lrt_runner")
    mod.__file__ = str(local)
    sys.modules["lrt_runner"] = mod
    exec(compile(data, str(local), "exec"), mod.__dict__)   # exactly the bytes that were verified
    return mod


_lrt_runner().run(globals(), data_ref=DATA_REF, manifest_sha256=MANIFEST_SHA256)